# 00. 환경 설정과 파일 인벤토리

## 학습 목표

1. 분석 전에 원본 경로와 실행 환경을 고정합니다.
2. CT와 mask가 사례별로 정확히 1:1 대응하는지 확인합니다.
3. 제공된 체크섬이 있으면 파일 손상을 검사합니다.

파일 수가 맞지 않거나 손상된 상태에서 통계 검사를 시작하면 이후 결과를 신뢰할 수 없습니다.
이 노트북은 원본을 **읽기 전용**으로 사용하고 결과만 `results/`에 기록합니다.

In [4]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    """VS Code가 프로젝트 폴더/노트북 폴더 중 어디를 cwd로 잡아도 동작하게 합니다."""
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "project_config.json").is_file() and (candidate / "ctqa_utils.py").is_file():
            return candidate
    raise FileNotFoundError("project_config.json이 있는 CT_Data_Quality_Notebooks 폴더를 VS Code에서 여세요.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from ctqa_utils import *

CONFIG = json.loads((PROJECT_ROOT / "project_config.json").read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"])
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("프로젝트:", PROJECT_ROOT)
print("데이터:", DATA_ROOT)
print("결과 저장:", RESULTS)

프로젝트: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합
데이터: C:\Users\duddl\OneDrive\바탕 화면\CTdata
결과 저장: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results


## 1. 데이터셋 발견과 CT-mask pairing

파일 확장자가 아니라 `.nii`/`.nii.gz`를 제거한 사례 ID로 대응합니다. 누락된 CT나 mask는 학습 샘플로 사용하면 안 됩니다.

In [5]:
datasets = discover_datasets(DATA_ROOT)
inventory_rows = []
pair_rows = []
source_summaries = []

for source, folders in datasets.items():
    images = list_nifti(folders["images"])
    masks = list_nifti(folders["masks"])
    pairs, images_without_mask, masks_without_image = pair_nifti(folders["images"], folders["masks"])

    for role, files in (("ct", images), ("mask", masks)):
        for path in files:
            inventory_rows.append({
                "source": source,
                "role": role,
                "case_id": nifti_stem(path),
                "path": str(path),
                "bytes": path.stat().st_size,
            })

    for case_id, image_path, mask_path in pairs:
        pair_rows.append({
            "source": source,
            "case_id": case_id,
            "group_id": bhsd_group_id(case_id) if source == "BHSD" else case_id,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
        })

    source_summaries.append({
        "source": source,
        "ct_files": len(images),
        "mask_files": len(masks),
        "paired_cases": len(pairs),
        "ct_without_mask": len(images_without_mask),
        "mask_without_ct": len(masks_without_image),
    })

inventory_df = pd.DataFrame(inventory_rows)
pair_manifest = pd.DataFrame(pair_rows)
source_summary = pd.DataFrame(source_summaries)

inventory_df.to_csv(RESULTS / "00_file_inventory.csv", index=False, encoding="utf-8-sig")
pair_manifest.to_csv(RESULTS / "00_pair_manifest.csv", index=False, encoding="utf-8-sig")
source_summary.to_csv(RESULTS / "00_inventory_summary.csv", index=False, encoding="utf-8-sig")

print(source_summary.to_string(index=False))
print(f"\n총 paired volume: {len(pair_manifest)}")

          source  ct_files  mask_files  paired_cases  ct_without_mask  mask_without_ct
            BHSD       192         192           192                0                0
PhysioNet_CT_ICH        75          75            75                0                0

총 paired volume: 267


## 2. PhysioNet 공식 SHA256 검증

체크섬은 파일 내용이 배포 이후 달라지지 않았는지 검사합니다. BHSD 로컬 폴더에는 기준 체크섬이 없으므로 PhysioNet만 자동 검증합니다.

데이터가 느린 디스크에 있으면 이 셀은 시간이 걸릴 수 있습니다. `project_config.json`의 `verify_physionet_sha256`을 `false`로 바꾸면 생략할 수 있습니다.

In [6]:
checksum_path = DATA_ROOT / "PhysioNet_CT_ICH" / "SHA256SUMS.txt"
if CONFIG.get("verify_physionet_sha256", True):
    checksum_df = pd.DataFrame(verify_sha256_manifest(checksum_path))
    checksum_df.to_csv(RESULTS / "00_physionet_sha256.csv", index=False, encoding="utf-8-sig")
    print("체크섬 항목:", len(checksum_df))
    print("파일 누락:", int((~checksum_df["exists"]).sum()))
    print("해시 불일치:", int((~checksum_df["match"]).sum()))
else:
    checksum_df = pd.DataFrame(columns=["relative_path", "exists", "match"])
    print("설정에 따라 SHA256 검증을 생략했습니다.")

체크섬 항목: 156
파일 누락: 0
해시 불일치: 0


## 3. 단계 판정

통과 조건은 CT-mask 누락이 0건이고, 제공된 체크섬을 검사한 경우 불일치가 0건인 것입니다.

In [7]:
pair_gate = bool((source_summary["ct_without_mask"] == 0).all() and (source_summary["mask_without_ct"] == 0).all())
checksum_gate = bool(checksum_df.empty or checksum_df["match"].all())
step_result = {
    "pairing_pass": pair_gate,
    "physionet_checksum_pass": checksum_gate,
    "total_pairs": int(len(pair_manifest)),
}
write_json(RESULTS / "00_step_result.json", step_result)
print(step_result)
assert pair_gate, "CT-mask 누락을 먼저 해결해야 합니다."
assert checksum_gate, "PhysioNet 체크섬 불일치를 먼저 해결해야 합니다."
print("\n00단계 PASS — 다음은 01 노트북입니다.")

{'pairing_pass': True, 'physionet_checksum_pass': True, 'total_pairs': 267}

00단계 PASS — 다음은 01 노트북입니다.
